# K-Fold Cross Validation

This notebook creates the stratified 5-fold cross-validation used to train and assess the VGG16-DCNN. Folds are built on the original training images, and every augmented copy is placed in the same fold as its original. This prevents an augmented version of a validation image from being used for training. Validation folds contain only original images.

## Prerequisites

- Python 3.10+
- Required libraries: `pandas`, `scikit-learn`
- Output of `OTSU_GWWHO_segmentation.ipynb` (`./outputs/04_segmented/train_segmented.csv`)

## Workflow

1. Load the segmented training set
2. Assign the original images to 5 stratified folds
3. Propagate the fold index to all augmented copies through `source_id`
4. Define the training / validation selection for each fold
5. Save the fold assignment

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

SEGMENTED_DIR = "./outputs/04_segmented"
FOLD_DIR = "./outputs/05_folds"
N_FOLDS = 5
RANDOM_SEED = 42

## Create Stratified Folds

Assigns each original image to one of the K = 5 folds; augmented images inherit the fold of their source image.

In [ ]:
# ==========================================================
# CREATE FOLDS
# ==========================================================

df = pd.read_csv(os.path.join(SEGMENTED_DIR, "train_segmented.csv"))
originals = df[df["is_augmented"] == 0].drop_duplicates("source_id").reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_of_source = {}
for k, (_, val_idx) in enumerate(skf.split(originals, originals["label"])):
    for sid in originals.loc[val_idx, "source_id"]:
        fold_of_source[sid] = k

df["fold"] = df["source_id"].map(fold_of_source).astype(int)

os.makedirs(FOLD_DIR, exist_ok=True)
df.to_csv(os.path.join(FOLD_DIR, "train_folds.csv"), index=False)

## Training / Validation Selection

For fold k: training = original and augmented images of the other four folds; validation = original images of fold k.

In [ ]:
# ==========================================================
# GET FOLD
# ==========================================================

def get_fold(df, k):
    train = df[df.fold != k].reset_index(drop=True)
    val = df[(df.fold == k) & (df.is_augmented == 0)].reset_index(drop=True)
    return train, val


for k in range(N_FOLDS):
    tr, va = get_fold(df, k)
    print(f"Fold {k}: training {len(tr):>7} images | validation {len(va):>6} images")